# Vehicle Detection and Tracking System

## Author: Based on Candy Calorie Counter by Evan Juras, EJ Technology Consultants

## Description:
This script uses YOLO vehicle detection model to locate and identify vehicles in recorded video files.
It uses a homemade StrongSORT tracker to track vehicles across frames and counts the total number
of vehicles detected on highway footage. The system processes video files and outputs annotated
video with vehicle counts and tracking information.

In [1]:
# Import necessary packages
import os
import sys
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
import math
import torch
import torchvision.transforms.functional as F
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
import logging

# Configure logging for debugging
logging.basicConfig(level=logging.INFO)

print("All required libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")

All required libraries imported successfully!
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA device: NVIDIA GeForce GTX 1070


In [2]:
# Homemade StrongSORT Tracker Implementation
class HomemadeStrongSORT:
    def __init__(self, max_disappeared=30, max_distance=100):
        """
        Initialize the StrongSORT tracker
        
        Args:
            max_disappeared (int): Maximum number of frames a track can be lost before deletion
            max_distance (float): Maximum distance for associating detections with existing tracks
        """
        self.next_object_id = 0
        self.objects = {}  # Dictionary to store tracked objects
        self.disappeared = {}  # Track how many frames each object has been missing
        self.max_disappeared = max_disappeared
        self.max_distance = max_distance
        
    def register(self, centroid, bbox, class_id, confidence):
        """Register a new object with given centroid, bounding box, and confidence"""
        self.objects[self.next_object_id] = {
            'centroid': centroid,
            'bbox': bbox,
            'class_id': class_id,
            'confidence': confidence,
            'trail': deque(maxlen=10)  # Store last 10 positions for trail
        }
        self.objects[self.next_object_id]['trail'].append(centroid)
        self.disappeared[self.next_object_id] = 0
        self.next_object_id += 1
        
    def deregister(self, object_id):
        """Remove an object from tracking"""
        del self.objects[object_id]
        del self.disappeared[object_id]
        
    def update(self, detections):
        """
        Update the tracker with new detections
        
        Args:
            detections (list): List of detections, each containing [x1, y1, x2, y2, conf, class_id]
        
        Returns:
            dict: Updated objects with tracking IDs
        """
        # If no detections, mark all existing objects as disappeared
        if len(detections) == 0:
            for object_id in list(self.disappeared.keys()):
                self.disappeared[object_id] += 1
                if self.disappeared[object_id] > self.max_disappeared:
                    self.deregister(object_id)
            return self.objects
        
        # Calculate centroids for all detections
        input_centroids = []
        input_bboxes = []
        input_classes = []
        input_confidences = []
        
        for detection in detections:
            x1, y1, x2, y2, conf, class_id = detection
            cx = int((x1 + x2) / 2.0)
            cy = int((y1 + y2) / 2.0)
            input_centroids.append((cx, cy))
            input_bboxes.append((x1, y1, x2, y2))
            input_classes.append(class_id)
            input_confidences.append(conf)
        
        # If no existing objects, register all detections as new objects
        if len(self.objects) == 0:
            for i in range(len(input_centroids)):
                self.register(input_centroids[i], input_bboxes[i], input_classes[i], input_confidences[i])
        else:
            # Compute distance matrix between existing objects and new detections
            object_centroids = [obj['centroid'] for obj in self.objects.values()]
            object_ids = list(self.objects.keys())
            
            # Calculate Euclidean distances
            D = np.linalg.norm(np.array(object_centroids)[:, np.newaxis] - np.array(input_centroids), axis=2)
            
            # Find the minimum distance associations
            rows = D.min(axis=1).argsort()
            cols = D.argmin(axis=1)[rows]
            
            # Keep track of used row and column indices
            used_row_indices = set()
            used_col_indices = set()
            
            # Update existing objects with closest detections
            for (row, col) in zip(rows, cols):
                if row in used_row_indices or col in used_col_indices:
                    continue
                
                if D[row, col] > self.max_distance:
                    continue
                
                # Update the object
                object_id = object_ids[row]
                self.objects[object_id]['centroid'] = input_centroids[col]
                self.objects[object_id]['bbox'] = input_bboxes[col]
                self.objects[object_id]['class_id'] = input_classes[col]
                self.objects[object_id]['confidence'] = input_confidences[col]  # Update confidence
                self.objects[object_id]['trail'].append(input_centroids[col])
                self.disappeared[object_id] = 0
                
                used_row_indices.add(row)
                used_col_indices.add(col)
            
            # Handle unmatched detections and objects
            unused_row_indices = set(range(0, D.shape[0])).difference(used_row_indices)
            unused_col_indices = set(range(0, D.shape[1])).difference(used_col_indices)
            
            # If there are more objects than detections, mark objects as disappeared
            if D.shape[0] >= D.shape[1]:
                for row in unused_row_indices:
                    object_id = object_ids[row]
                    self.disappeared[object_id] += 1
                    
                    if self.disappeared[object_id] > self.max_disappeared:
                        self.deregister(object_id)
            
            # If there are more detections than objects, register new objects
            else:
                for col in unused_col_indices:
                    self.register(input_centroids[col], input_bboxes[col], input_classes[col], input_confidences[col])
        
        return self.objects

print("HomemadeStrongSORT tracker class defined successfully!")

HomemadeStrongSORT tracker class defined successfully!


In [3]:
# Configuration Parameters
# Define path to model and other user variables
model_path = 'yolo11x.pt'  # Path to YOLO model (you can replace with your custom model)
input_video_path = ''  # Will be set dynamically - input video file path
output_video_path = 'vehicle_tracking_output.mp4'  # Output video file path
min_thresh = 0.30  # Minimum detection threshold
record_fps = 120  # FPS for output video

# Vehicle classes that we want to detect on highway (COCO dataset class IDs)
# You can modify this based on your custom model classes
vehicle_classes = {
    2: 'car',
    3: 'motorcycle', 
    5: 'bus',
    7: 'truck',
    1: 'bicycle'  # Sometimes bicycles are on highways too
}

# Class names for display (modify based on your model)
vehicle_display_names = {
    2: 'Car',
    3: 'Motorcycle',
    5: 'Bus', 
    7: 'Truck',
    1: 'Bicycle'
}

print("Configuration parameters set!")
print(f"Vehicle classes to detect: {list(vehicle_display_names.values())}")

Configuration parameters set!
Vehicle classes to detect: ['Car', 'Motorcycle', 'Bus', 'Truck', 'Bicycle']


In [1]:
# Helper Functions
def draw_trail(frame, trail_points, color, thickness=2):
    """Draw tracking trail for an object"""
    if len(trail_points) > 1:
        pts = np.array(list(trail_points), dtype=np.int32)
        pts = pts.reshape((-1, 1, 2))
        cv2.polylines(frame, [pts], False, color, thickness)

def draw_info_panel(frame, vehicle_counts, total_vehicles, fps=0):
    """Draw information panel with vehicle counts"""
    panel_height = 150
    panel_width = 400
    
    # Draw semi-transparent background
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (panel_width, panel_height), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
    
    # Draw border
    cv2.rectangle(frame, (10, 10), (panel_width, panel_height), (255, 255, 255), 2)
    
    y_offset = 35
    # Total vehicles
    cv2.putText(frame, f'Total Vehicles Tracked: {total_vehicles}', 
                (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    y_offset += 25
    
    # Individual vehicle counts
    for class_id, count in vehicle_counts.items():
        if class_id in vehicle_display_names and count > 0:
            vehicle_name = vehicle_display_names[class_id]
            cv2.putText(frame, f'{vehicle_name}: {count}', 
                        (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            y_offset += 20
    
    # FPS
    if fps > 0:
        cv2.putText(frame, f'FPS: {fps:.1f}', 
                    (20, panel_height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

def get_bbox_colors():
    """Get predefined colors for bounding boxes"""
    return [
        (255, 0, 0),     # Red
        (0, 255, 0),     # Green  
        (0, 0, 255),     # Blue
        (255, 255, 0),   # Yellow
        (255, 0, 255),   # Magenta
        (0, 255, 255),   # Cyan
        (128, 0, 128),   # Purple
        (255, 165, 0),   # Orange
        (255, 192, 203), # Pink
        (0, 128, 0)      # Dark Green
    ]

print("Helper functions defined successfully!")

Helper functions defined successfully!


In [5]:
# Main Vehicle Detection and Tracking Function
def process_video_with_tracking(input_path, output_path, model_path, min_confidence=0.3):
    """
    Process video with vehicle detection and tracking
    
    Args:
        input_path (str): Path to input video file
        output_path (str): Path to output video file  
        model_path (str): Path to YOLO model
        min_confidence (float): Minimum confidence threshold for detections
    """
    
    # Check if model file exists
    if not os.path.exists(model_path):
        print(f'ERROR: Model path {model_path} is invalid or model was not found.')
        return False
        
    # Check if input video exists
    if not os.path.exists(input_path):
        print(f'ERROR: Input video {input_path} was not found.')
        return False
    
    print(f"Loading YOLO model from: {model_path}")
    # Load the YOLO model
    model = YOLO(model_path, task='detect')
    
    print(f"Opening input video: {input_path}")
    # Initialize video capture
    cap = cv2.VideoCapture(input_path)
    
    if not cap.isOpened():
        print("ERROR: Could not open input video file.")
        return False
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video properties: {width}x{height} @ {fps} FPS, Total frames: {total_frames}")
    
    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Initialize tracker
    tracker = HomemadeStrongSORT(max_disappeared=30, max_distance=150)
    
    # Initialize counters
    frame_count = 0
    total_vehicles_ever_seen = set()  # Track unique vehicle IDs
    vehicle_counts = defaultdict(int)  # Count by vehicle type
    bbox_colors = get_bbox_colors()
    
    print("Starting video processing...")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        # Print progress every 100 frames
        if frame_count % 100 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Processing frame {frame_count}/{total_frames} ({progress:.1f}%)")
        
        # Run YOLO detection
        results = model(frame, verbose=False)
        detections = results[0].boxes
        
        # Process detections - filter for vehicle classes only
        current_detections = []
        
        if detections is not None and len(detections) > 0:
            for i in range(len(detections)):
                # Get bounding box coordinates
                xyxy_tensor = detections[i].xyxy.cpu()
                xyxy = xyxy_tensor.numpy().squeeze()
                x1, y1, x2, y2 = xyxy.astype(int)
                
                # Get class and confidence
                class_id = int(detections[i].cls.item())
                confidence = detections[i].conf.item()
                
                # Only process vehicle classes with sufficient confidence
                if class_id in vehicle_classes and confidence > min_confidence:
                    current_detections.append([x1, y1, x2, y2, confidence, class_id])
        
        # Update tracker with current detections
        tracked_objects = tracker.update(current_detections)
        
        # Draw tracking results
        current_frame_counts = defaultdict(int)
        
        for object_id, obj_data in tracked_objects.items():
            total_vehicles_ever_seen.add(object_id)
            
            x1, y1, x2, y2 = obj_data['bbox']
            class_id = obj_data['class_id']
            centroid = obj_data['centroid']
            trail = obj_data['trail']
            
            # Count vehicles in current frame
            current_frame_counts[class_id] += 1
            
            # Get color for this object
            color = bbox_colors[object_id % len(bbox_colors)]
            
            # Draw bounding box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            
            # Draw tracking trail
            if len(trail) > 1:
                draw_trail(frame, trail, color, thickness=2)
            
            # Draw object ID and class label
            if class_id in vehicle_display_names:
                label = f'ID:{object_id} {vehicle_display_names[class_id]} {int(confidence * 100)}%'
                label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                
                # Draw label background
                cv2.rectangle(frame, (int(x1), int(y1) - label_size[1] - 10), 
                             (int(x1) + label_size[0], int(y1)), color, -1)
                
                # Draw label text
                cv2.putText(frame, label, (int(x1), int(y1) - 5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                # Draw center point
                cv2.circle(frame, centroid, 4, color, -1)
        
        # Update overall vehicle counts (keep track of maximum seen for each type)
        for class_id, count in current_frame_counts.items():
            vehicle_counts[class_id] = max(vehicle_counts[class_id], count)
        
        # Draw information panel(optional)
        #draw_info_panel(frame, vehicle_counts, len(total_vehicles_ever_seen), 
                       #fps if frame_count > 0 else 0)
        
        # Write frame to output video
        out.write(frame)
    
    # Clean up
    cap.release()
    out.release()
    
    print(f"\\nVideo processing completed!")
    print(f"Output saved to: {output_path}")
    print(f"Total unique vehicles tracked: {len(total_vehicles_ever_seen)}")
    print("Vehicle type breakdown:")
    for class_id, count in vehicle_counts.items():
        if class_id in vehicle_display_names:
            print(f"  {vehicle_display_names[class_id]}: {count}")
    
    return True

print("Main processing function defined successfully!")

Main processing function defined successfully!


In [6]:
# Example Usage - Process a Video File
# Set your input video path here
input_video = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found_หิวโว้ยยย.MP4"  # Change this to your video file path
output_video = r"C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found_2ย.MP4"  # Output will be saved with this name
# Check if the input video file exists
if os.path.exists(input_video):
    print(f"Found input video: {input_video}")
    
    # Process the video
    success = process_video_with_tracking(
        input_path=input_video,
        output_path=output_video, 
        model_path=model_path,
        min_confidence=min_thresh
    )
    
    if success:
        print("\\n✅ Video processing completed successfully!")
        print(f"📁 Output video saved as: {output_video}")
        print("\\n🚗 The output video includes:")
        print("   • Vehicle detection and tracking")
        print("   • Unique vehicle ID assignment") 
        print("   • Vehicle counting by type")
        print("   • Tracking trails")
        print("   • Real-time statistics overlay")
    else:
        print("❌ Video processing failed!")
        
else:
    print(f"❌ Input video not found: {input_video}")
    print("\\n📋 Available video files in workspace:")
    
    # List available video files
    video_extensions = ['.mp4', '.avi', '.mkv', '.mov', '.MOV', '.MP4']
    video_files = []
    
    for root, dirs, files in os.walk(r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder"):
        for file in files:
            if any(file.lower().endswith(ext.lower()) for ext in video_extensions):
                video_files.append(os.path.join(root, file))
    
    for i, video_file in enumerate(video_files[:10]):  # Show first 10 videos
        print(f"   {i+1}. {video_file}")
    
    if len(video_files) > 10:
        print(f"   ... and {len(video_files) - 10} more videos")
    
    print("\\n💡 To use this code:")
    print("   1. Copy one of the video file paths above") 
    print("   2. Update the 'input_video' variable in this cell")
    print("   3. Run this cell again")

Found input video: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found_หิวโว้ยยย.MP4
Loading YOLO model from: yolo11x.pt
Opening input video: C:\Users\Taber\Desktop\Nutt_KMUTT_folder\7\KEN-5895\FILE240403-085108F_lane_change_found_หิวโว้ยยย.MP4
Video properties: 2560x1440 @ 30 FPS, Total frames: 1800
Starting video processing...
Processing frame 100/1800 (5.6%)
Processing frame 200/1800 (11.1%)
Processing frame 300/1800 (16.7%)
Processing frame 400/1800 (22.2%)
Processing frame 500/1800 (27.8%)
Processing frame 600/1800 (33.3%)
Processing frame 700/1800 (38.9%)
Processing frame 800/1800 (44.4%)
Processing frame 900/1800 (50.0%)
Processing frame 1000/1800 (55.6%)
Processing frame 1100/1800 (61.1%)
Processing frame 1200/1800 (66.7%)
Processing frame 1300/1800 (72.2%)
Processing frame 1400/1800 (77.8%)
Processing frame 1500/1800 (83.3%)
Processing frame 1600/1800 (88.9%)
Processing frame 1700/1800 (94.4%)
Processing frame 1800/1800 (100.0%)
\nVideo pro

In [35]:
# Alternative Usage - Process Different Video Files
# You can try different videos from your workspace

# Example with different video files from your folders:
alternative_videos = [
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\2\737505456.564599.mp4",
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\3\739790513.565241.mp4", 
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\4\1130327--910-G6.MOV",
    r"c:\Users\Taber\Desktop\Nutt_KMUTT_folder\5\737795575.325078.mp4"
]

# Uncomment and modify one of these to process a different video:
# input_video = alternative_videos[0]  # Change index 0 to 1, 2, 3 for different videos
# output_video = "vehicle_tracking_output_2.mp4"

# Or manually set your video path:
# input_video = r"path\to\your\video\file.mp4"
# output_video = "your_output_name.mp4"

print("Alternative video options available above.")
print("Uncomment and modify the lines above to process different videos.")

Alternative video options available above.
Uncomment and modify the lines above to process different videos.


In [36]:
# Customization and Troubleshooting

# If you have your own trained model, update the model_path:
# model_path = r"path\to\your\custom\model.pt"

# Adjust tracking parameters if needed:
def create_custom_tracker(max_disappeared=30, max_distance=150):
    """
    Create tracker with custom parameters
    
    Args:
        max_disappeared (int): Frames before removing lost tracks (higher = more persistent tracking)
        max_distance (float): Max distance for track association (higher = more flexible matching)
    """
    return HomemadeStrongSORT(max_disappeared=max_disappeared, max_distance=max_distance)

# Troubleshooting Tips:
print("🔧 Troubleshooting Guide:")
print("1. If too many false detections: Increase min_thresh (try 0.4 or 0.5)")
print("2. If missing vehicles: Decrease min_thresh (try 0.2 or 0.25)")
print("3. If tracking is jumpy: Increase max_distance in tracker (try 200)")
print("4. If tracks persist too long: Decrease max_disappeared (try 15-20)")
print("5. If using custom model: Update vehicle_classes dictionary with your class IDs")

# Performance Tips:
print("\\n⚡ Performance Tips:")
print("1. For faster processing: Use smaller YOLO model (yolo11n.pt vs yolo11s.pt)")
print("2. For better accuracy: Use larger YOLO model (yolo11m.pt or yolo11l.pt)")
print("3. Adjust input video resolution if needed")
print("4. Consider processing every Nth frame for faster results (modify frame skip)")

# Custom vehicle classes (if using your own model):
# custom_vehicle_classes = {
#     0: 'car',
#     1: 'truck', 
#     2: 'bus',
#     3: 'motorcycle'
# }
# 
# custom_display_names = {
#     0: 'Car',
#     1: 'Truck',
#     2: 'Bus', 
#     3: 'Motorcycle'
# }

print("\\nCustomization options and troubleshooting tips displayed above.")

🔧 Troubleshooting Guide:
1. If too many false detections: Increase min_thresh (try 0.4 or 0.5)
2. If missing vehicles: Decrease min_thresh (try 0.2 or 0.25)
3. If tracking is jumpy: Increase max_distance in tracker (try 200)
4. If tracks persist too long: Decrease max_disappeared (try 15-20)
5. If using custom model: Update vehicle_classes dictionary with your class IDs
\n⚡ Performance Tips:
1. For faster processing: Use smaller YOLO model (yolo11n.pt vs yolo11s.pt)
2. For better accuracy: Use larger YOLO model (yolo11m.pt or yolo11l.pt)
3. Adjust input video resolution if needed
4. Consider processing every Nth frame for faster results (modify frame skip)
\nCustomization options and troubleshooting tips displayed above.


## Additional Features:add optical flow